# MyDream Expanded Sequence Model Comparison

Run the current post-GRU architecture comparison in Colab and review the alarm-window results in tables and plots.

This notebook compares the existing selected GRU against TCN, Transformer, and CNN+GRU. It also reruns the comparison at the deployment-relevant `0.55` threshold, which is not included in the analyzer default threshold set.

## 1. Runtime Setup

Use a Colab runtime with GPU when available. The generated outputs remain under `PROFILE_ROOT`, outside Git.

In [ ]:
!pip -q install tensorflow pandas matplotlib seaborn

## 2. Load The Git-Linked Code Folder

This cell clones the project on a fresh Colab runtime or pulls the latest code when the folder is already present.

In [ ]:
from pathlib import Path
import subprocess

CODE_ROOT = Path('/content/mydream-training-evaluation')
REPOSITORY_URL = 'https://github.com/sfpahsdev-mydream/mydream-training-evaluation.git'

if (CODE_ROOT / '.git').exists():
    subprocess.run(['git', '-C', str(CODE_ROOT), 'pull', '--ff-only'], check=True)
else:
    subprocess.run(['git', 'clone', REPOSITORY_URL, str(CODE_ROOT)], check=True)

assert (CODE_ROOT / 'run_sequence_experiment_matrix.py').exists(), CODE_ROOT
print('Code ready:', CODE_ROOT)

## 3. Configure And Verify Inputs

Set `INCLUDE_TABULAR = True` only after `model_tabular_tflite/alarm_predictions_long.csv` exists for this same profile.

In [ ]:
from pathlib import Path

PROFILE_ROOT = Path('/content/mydream_latest/out/latest_fixed_wake_policy')
INCLUDE_TABULAR = False
THRESHOLDS = [0.4, 0.5, 0.55, 0.6]

required_paths = [
    PROFILE_ROOT / 'sequence_60m',
    PROFILE_ROOT / 'sequence_60m_alarm',
    PROFILE_ROOT / 'sequence_model_gru' / 'alarm_predictions_long.csv',
]
for path in required_paths:
    assert path.exists(), f'Missing required input: {path}'

tabular_predictions = PROFILE_ROOT / 'model_tabular_tflite' / 'alarm_predictions_long.csv'
if INCLUDE_TABULAR:
    assert tabular_predictions.exists(), f'Missing tabular predictions: {tabular_predictions}'

print('Profile root:', PROFILE_ROOT)
print('Include tabular:', INCLUDE_TABULAR)
print('Thresholds:', THRESHOLDS)

## 4. Train Expanded Candidates

This trains TCN, Transformer, and CNN+GRU, then performs the script's standard comparison. Existing candidate prediction outputs are reused.

In [ ]:
import subprocess
import sys

matrix_command = [
    sys.executable,
    str(CODE_ROOT / 'run_sequence_experiment_matrix.py'),
    '--profile-root', str(PROFILE_ROOT),
    '--experiment-set', 'expanded',
    '--skip-existing',
]
if not INCLUDE_TABULAR:
    matrix_command.append('--no-tabular-model')

print('Running:', ' '.join(matrix_command))
subprocess.run(matrix_command, cwd=CODE_ROOT, check=True)

## 5. Recompute Comparable Threshold Results

The current operating candidate uses threshold `0.55`, so this cell writes a compact comparison using `0.4`, `0.5`, `0.55`, and `0.6` for every architecture.

In [ ]:
comparison_root = PROFILE_ROOT / 'sequence_experiments' / 'expanded'
comparison_output = comparison_root / 'alarm_failure_comparison_selected_thresholds'

model_dirs = [
    PROFILE_ROOT / 'sequence_model_gru',
    comparison_root / 'tcn64_dense32_dropout00',
    comparison_root / 'transformer64_dense32_dropout10',
    comparison_root / 'cnn32_gru64_dense32_dropout00',
]
if INCLUDE_TABULAR:
    model_dirs.insert(0, PROFILE_ROOT / 'model_tabular_tflite')

for model_dir in model_dirs:
    predictions = model_dir / 'alarm_predictions_long.csv'
    assert predictions.exists(), f'Missing model prediction output: {predictions}'

analysis_command = [sys.executable, str(CODE_ROOT / 'analyze_alarm_failures.py')]
for model_dir in model_dirs:
    analysis_command += ['--model-dir', str(model_dir)]
for threshold in THRESHOLDS:
    analysis_command += ['--threshold', str(threshold), '--focus-threshold', str(threshold)]
analysis_command += ['--output-dir', str(comparison_output)]

print('Running:', ' '.join(analysis_command))
subprocess.run(analysis_command, cwd=CODE_ROOT, check=True)
print('Comparison output:', comparison_output)

## 6. View Threshold Comparison

Prefer higher `deep_success` and `success_per_smart`, while keeping `strong_fail` low. The current deployment reference threshold is `0.55`.

In [ ]:
import pandas as pd
from IPython.display import display

summary_path = comparison_output / 'model_comparison_summary.csv'
summary = pd.read_csv(summary_path)
summary = summary.sort_values(['threshold', 'strong_fail', 'deep_success'], ascending=[True, True, False])

display(summary.reset_index(drop=True))

reference = summary[summary['threshold'].eq(0.55)].copy()
reference = reference.sort_values(['strong_fail', 'deep_success', 'success_per_smart'], ascending=[True, False, False])
print('Threshold 0.55 comparison')
display(reference.reset_index(drop=True))

## 7. Visual Comparison

Use the first chart to judge the `deep_success` versus `strong_fail` tradeoff. A candidate materially better than GRU should move toward more success with fewer failures under the same threshold.

In [ ]:
import matplotlib.pyplot as plt
import seaborn as sns

sns.set_theme(style='whitegrid')
fig, axes = plt.subplots(1, 2, figsize=(15, 6))

sns.scatterplot(
    data=summary,
    x='strong_fail',
    y='deep_success',
    hue='model',
    style='threshold',
    s=120,
    ax=axes[0],
)
axes[0].set_title('Alarm Tradeoff By Threshold')
axes[0].set_xlabel('Strong Fail (lower is better)')
axes[0].set_ylabel('Deep Success (higher is better)')

plot_data = reference.melt(
    id_vars=['model', 'threshold'],
    value_vars=['deep_success', 'strong_fail'],
    var_name='metric',
    value_name='count',
)
sns.barplot(data=plot_data, x='model', y='count', hue='metric', errorbar=None, ax=axes[1])
axes[1].set_title('Threshold 0.55 Counts By Model')
axes[1].set_xlabel('Model')
axes[1].tick_params(axis='x', rotation=20)

plt.tight_layout()
plt.show()

## 8. Export Comparison Files

Download the compact comparison output when the Colab runtime is temporary. Generated model artifacts and datasets should remain outside Git.

In [ ]:
import shutil

archive_path = shutil.make_archive(
    str(comparison_output),
    'zip',
    root_dir=comparison_output,
)
print('Created:', archive_path)

# Uncomment in Colab to download the result summary archive.
# from google.colab import files
# files.download(archive_path)